In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -qU \
langchain \
langchain-community \
langchain-huggingface \
langchain-groq \
faiss-cpu \
sentence-transformers \
pypdf

In [3]:
import langchain
import langchain_core
import langchain_community

print("LangChain:", langchain.__version__)
print("LangChain Core:", langchain_core.__version__)
print("LangChain Community:", langchain_community.__version__)

LangChain: 1.3.12
LangChain Core: 1.4.9
LangChain Community: 0.4.2


/tmp/ipykernel_1537/373314028.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  import langchain_community


In [4]:
!pip install -q langchain-text-splitters

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
from langchain_community.document_loaders import PyPDFLoader

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

print("✅ All imports successful!")

/tmp/ipykernel_5092/793319003.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


✅ All imports successful!


In [7]:
from google.colab import files

uploaded = files.upload()

Saving Generative AI (1).pdf to Generative AI (1).pdf


In [9]:
pdf_name = list(uploaded.keys())[0]

loader = PyPDFLoader(pdf_name)

documents = loader.load()

#Check Number of pages

print("Total Pages:", len(documents))

Total Pages: 50


In [11]:
#Display the First Page
print(documents[0].page_content)

Generative AI for Everyone
Dr. Jawwad A Shamsi
Professor and Dean
FAST NUCES
Jawwad.shamsi@nu.edu.pk


In [14]:
#Check Metadata
print(documents[0].metadata)


{'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Generative AI.pptx', 'moddate': '2026-02-07T19:05:37+05:00', 'source': 'Generative AI (1).pdf', 'total_pages': 50, 'page': 0, 'page_label': '1'}


In [16]:
#View Multiple pages
for i in range(3):
    print("="*100)
    print(f"PAGE {i+1}")
    print("="*100)

    print(documents[i].page_content[:500])

    print()

PAGE 1
Generative AI for Everyone
Dr. Jawwad A Shamsi
Professor and Dean
FAST NUCES
Jawwad.shamsi@nu.edu.pk

PAGE 2
SYSLAB (My Research Group at 
FAST-NUCES)
• Systems Research Laboratory
• Developing Capable Systems through integration 
• Artificial Intelligence
• Cloud Computing
• Security
• High Performance Computing
• Health
• Education
 Jawwad A Shamsi - ICONICS 2022

PAGE 3
What is Generative AI
Generate new Context
Text
Images
Documents
Videos
How?



# Module 3: Text Chunking

## Objective
Split the extracted PDF text into smaller chunks to improve retrieval accuracy.

### Why is chunking needed?

Large Language Models cannot efficiently process an entire PDF at once. We split the document into overlapping chunks so that relevant information can be retrieved efficiently during question answering.

**Parameters**
- chunk_size = 500
- chunk_overlap = 100

In [11]:
#Text Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print("Total Chunks:", len(chunks))

Total Chunks: 80


In [13]:
#View First Chunk
print(chunks[0].page_content)

Professor Mayur Naik
CIS 7000 - Fall 2024
Introduction
Slides adapted in part from Stanford CS25: Transformers United V4 (Spring’24).


In [14]:
#View Second Chunk
print("=" *80)
print(chunks[1].page_content)

● The Turing Test
● Overview of LLMs
○ How do LLMs work, What LLMs can do, Limitations of LLMs, 
What is the future
● Course Logistics
Today’s Agenda


📘 Module 4 – Generate Embeddings

In [15]:
#Step 1: Load the Embedding Model
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("✅ Embedding model loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded successfully!


In [16]:
#Step 2: Test the Embedding Model
sample_text = "What is Artificial Intelligence?"

embedding = embedding_model.embed_query(sample_text)

print("Type:", type(embedding))
print("Embedding Length:", len(embedding))
print("First 10 Values:\n")
print(embedding[:10])

Type: <class 'list'>
Embedding Length: 384
First 10 Values:

[-0.015781095251441002, 0.014908617362380028, 0.00909555796533823, 0.03131742775440216, -0.02170826494693756, -0.04852483794093132, 0.04887937009334564, 0.028673212975263596, -0.02559719979763031, 0.05404298007488251]


In [17]:
#Step 3: Generate Embeddings for All Chunks
texts = [chunk.page_content for chunk in chunks]

print("Total Chunks:", len(texts))

Total Chunks: 80


In [18]:
#Step 4: Create the FAISS Vector Database
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

print("✅ FAISS Vector Database Created Successfully!")

✅ FAISS Vector Database Created Successfully!


In [19]:
#Step 5: Save the Vector Database
vectorstore.save_local("faiss_index")

print("✅ Vector database saved successfully!")

✅ Vector database saved successfully!


In [20]:
#Step 6: Test Similarity Search
query = "What is Generative AI?"

results = vectorstore.similarity_search(query, k=3)

print(f"Retrieved {len(results)} Relevant Chunks\n")

Retrieved 3 Relevant Chunks



In [21]:
#Step 7: Display the Retrieved Chunks
for i, doc in enumerate(results, start=1):
    print("=" * 80)
    print(f"Result {i}")
    print("=" * 80)

    print(doc.page_content[:500])

    print("\nMetadata:")
    print(doc.metadata)

    print("\n")

Result 1
Risks from Synthetic Data
I. Shumailov et al. AI Models Collapse When Trained on Recursively Generated Data. 2024.

Metadata:
{'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Lecture 1 - Introduction and Course Overview', 'moddate': '2026-02-14T18:59:54+05:00', 'source': 'Lecture 1 - Introduction and Course Overview.pdf', 'total_pages': 64, 'page': 42, 'page_label': '43'}


Result 2
● Application Domains: we won’t dive into specific domains like NLP , Vision, or Robotics
● Theory: limited to mathematical concepts needed to understand and implement techniques
● Classical ML: we won’t cover classical ML approaches that predate LLMs
● AI Application Dev: we won’t teach you the AI dev stack or how to build enterprise AI apps

Metadata:
{'producer': 'PyPDF', 'creator': 'Google', 'creationdate': '', 'title': 'Lecture 1 - Introduction and Course Overview', 'moddate': '2026-02-14T18:59:54+05:00', 'source': 'Lecture 1 - Introduction and Course Overview.pdf', 'to